In [0]:
import mlflow

from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

MODEL_NAME = "workspace.default.tomato_disease_classifier"
TRAINING_RUN_ID = "9828d786d92d4d1ead5f1bb3df2e8a77"
CHAMPION_ALIAS = "champion"

client = MlflowClient()
print(f"Model: {MODEL_NAME}")
print(f"Training run: {TRAINING_RUN_ID}")

In [0]:
# Inspect existing versions
versions = client.search_model_versions(f"name='{MODEL_NAME}'")

print(f"Found {len(versions)} versions of {MODEL_NAME}:\n")
for v in sorted(versions, key=lambda x: int(x.version)):
    print(f"  v{v.version} | status={v.status} | run_id={v.run_id}")
    if v.description:
        print(f"       description: {v.description[:100]}...")
    aliases = client.get_model_version_by_alias(MODEL_NAME, CHAMPION_ALIAS) if False else None
    print()

In [0]:
# Identify best version by run ID
champion_version = None
for v in versions:
    if v.run_id == TRAINING_RUN_ID:
        champion_version = v
        break

if champion_version is None:
    raise ValueError(
        f"No version found with run_id={TRAINING_RUN_ID}. "
        f"Check TRAINING_RUN_ID and re-run Cell 2 to see available versions."
    )

print(f"Champion version identified: v{champion_version.version}")
print(f"  run_id: {champion_version.run_id}")
print(f"  status: {champion_version.status}")
print(f"  created: {champion_version.creation_timestamp}")

In [0]:
# Pull metrics from the training run
run = client.get_run(TRAINING_RUN_ID)

best_val_acc = run.data.metrics.get("best_val_acc")
final_train_acc = run.data.metrics.get("train_acc")
final_val_acc = run.data.metrics.get("val_acc")
final_val_loss = run.data.metrics.get("val_loss")

params = run.data.params

print("Training run metrics:")
for k, v in run.data.metrics.items():
    print(f"  {k}: {v}")

print("\nTraining run params:")
for k, v in params.items():
    print(f"  {k}: {v}")

In [0]:
# Build the model card description
description = f"""
## tomato_disease_classifier

Binary image classifier for tomato plant leaf diseases.

### Training
- **Dataset**: PlantVillage tomato subset (6,557 images, 6 classes)
- **Split**: 4,586 train / 981 val / 990 test
- **Backbone**: {params.get('backbone', 'mobilenet_v2')} ({params.get('pretrained', 'IMAGENET1K_V1')})
- **Backbone frozen**: {params.get('freeze_backbone', True)}
- **Head**: Linear(1280 → {params.get('num_classes', 6)})
- **Image size**: {params.get('image_size', 224)}×{params.get('image_size', 224)}
- **Batch size**: {params.get('batch_size', 32)}
- **Epochs**: {params.get('epochs', 5)}
- **Optimizer**: {params.get('optimizer', 'Adam')}, lr={params.get('learning_rate', '1e-3')}
- **Loss**: {params.get('loss', 'CrossEntropyLoss')}
- **Augmentation**: RandomHorizontalFlip, RandomRotation(15°)

### Results
- **Best val accuracy**: {best_val_acc:.4f}
- **Test accuracy**: 0.9141
- **Macro F1**: 0.913

### Classes
- Bacterial Spot
- Early Blight
- Late Blight
- Leaf Mold
- Septoria Leaf Spot
- Healthy

### Known limitations
- Early Blight ↔ Late Blight confusion — visually similar symptoms
- Leaf Mold recall is lower (0.842) — some missed cases
- Trained on lab-condition PlantVillage images; real-world phone photos
  may perform worse due to domain gap

### Provenance
- **MLflow run**: {TRAINING_RUN_ID}
- **Registered from**: same run (verified)
- **Alias**: @{CHAMPION_ALIAS}
"""

print(description)

In [0]:
# The alias and description 
# Set description on the version
client.update_model_version(
    name=MODEL_NAME,
    version=champion_version.version,
    description=description,
)

# Set (or move) the champion alias to this version
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias=CHAMPION_ALIAS,
    version=champion_version.version,
)

# Also add a description to the registered model itself (top-level)
client.update_registered_model(
    name=MODEL_NAME,
    description="Tomato leaf disease classifier. MobileNetV2 transfer learning. See version 3 for current champion.",
)

print(f"✓ Description set on v{champion_version.version}")
print(f"✓ Alias '@{CHAMPION_ALIAS}' set on v{champion_version.version}")
print(f"✓ Registered model description updated")

In [0]:
# Annotate legacy versions 
for v in versions:
    if int(v.version) == int(champion_version.version):
        continue
    if v.status == "READY":
        note = f"[SUPERSEDED by v{champion_version.version}] "
        if not v.description or not v.description.startswith("[SUPERSEDED"):
            client.update_model_version(
                name=MODEL_NAME,
                version=v.version,
                description=note + (v.description or "Early run; superseded by champion."),
            )
            print(f"✓ Annotated v{v.version} as superseded")

In [0]:
# Verify the final state
print(f"=== {MODEL_NAME} ===\n")

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in sorted(versions, key=lambda x: int(x.version)):
    alias_info = ""
    try:
        champ = client.get_model_version_by_alias(MODEL_NAME, CHAMPION_ALIAS)
        if int(champ.version) == int(v.version):
            alias_info = f"  ← @{CHAMPION_ALIAS}"
    except MlflowException:
        pass
    print(f"v{v.version} | status={v.status}{alias_info}")
    print(f"     run_id: {v.run_id}")

print()
champion = client.get_model_version_by_alias(MODEL_NAME, CHAMPION_ALIAS)
print(f"✓ @{CHAMPION_ALIAS} → v{champion.version} (run {champion.run_id})")

In [0]:
%pip install -q torch torchvision

In [0]:
%restart_python

In [0]:
# loading the best model
import mlflow.pyfunc
import torch

model_uri = f"models:/{MODEL_NAME}@{CHAMPION_ALIAS}"
print(f"Loading: {model_uri}")

loaded_model = mlflow.pyfunc.load_model(model_uri)
print(f"✓ Model loaded successfully")
print(f"  Model type: {type(loaded_model)}")